# Evaluation — Italian WITS

Full evaluation with traditional metrics, abstraction metrics,
and LLM-as-Judge scoring.

In [ ]:
!pip install -e ../..

In [ ]:
from huggingface_hub import login
login()

## Configuration

In [ ]:
from sm_sip.config import SigExtConfig, InferenceConfig, EvalConfig

sigext_config = SigExtConfig.from_preset("it", "10k-60t")
inference_config = InferenceConfig(lang="it", quantization="8bit", prompt_type="source_aware")
eval_config = EvalConfig(lang="it", judge_model_id="Qwen/Qwen2.5-14B-Instruct")

## Load Data & Models

In [ ]:
from sm_sip.data import get_test_data
from sm_sip.models import load_sigext_model, load_llm, create_summary_chain, create_judge_chain, preprocess_dataset
from sm_sip.prompts import get_summary_prompt, get_judge_prompt

test_data = get_test_data(lang="it", num_samples=100, skip_samples=sigext_config.skip_samples)
sigext_model, sigext_tokenizer = load_sigext_model(sigext_config.model_id)
processed_data = preprocess_dataset(test_data, sigext_model, sigext_tokenizer, lang="it")

# Summary chain
llm_model, llm_tokenizer, gen_pipe = load_llm(inference_config.llm_model_id, inference_config.quantization)
summary_chain = create_summary_chain(gen_pipe, get_summary_prompt("it", "source_aware"))

# Judge chain
judge_prompt = get_judge_prompt("unified")
judge_chain = create_judge_chain(gen_pipe, judge_prompt)

## Run Enhanced Evaluation

In [ ]:
from sm_sip.pipelines import run_enhanced_evaluation

metrics, samples = run_enhanced_evaluation(
    processed_data,
    summary_chain,
    judge_chain=judge_chain,
    lang="it",
)

print("\n=== RESULTS ===")
for key, val in metrics.items():
    print(f"  {key}: {val['mean']:.4f} ± {val['std']:.4f}")

## Save Results

In [ ]:
from sm_sip.utils.io import save_results
from datetime import datetime

save_results({
    "run_info": {
        "timestamp": datetime.now().isoformat(),
        "sigext_model": sigext_config.model_id,
        "prompt_type": inference_config.prompt_type,
        "num_samples": len(samples),
    },
    "metrics": metrics,
    "samples": samples,
}, "results/italian/eval_enhanced.json")

print("Results saved!")

## Cleanup

In [ ]:
from sm_sip.utils.gpu import clear_gpu_memory
del llm_model, llm_tokenizer, gen_pipe, sigext_model
clear_gpu_memory()